In [1]:
from datasets import load_dataset

dataset = load_dataset("GBaker/MedQA-USMLE-4-options")

In [2]:
# dataset["train"][0]
dataset.save_to_disk("datasets/raw/medqa")

Saving the dataset (0/1 shards):   0%|          | 0/10178 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/1273 [00:00<?, ? examples/s]

In [3]:
dataset["train"].to_json(
    "datasets/processed/train.jsonl",
    orient="records",
    lines=True
)

Creating json from Arrow format:   0%|          | 0/11 [00:00<?, ?ba/s]

15751834

In [4]:
dataset["test"].to_json(
    "datasets/processed/test.jsonl",
    orient="records",
    lines=True
)

Creating json from Arrow format:   0%|          | 0/2 [00:00<?, ?ba/s]

2017628

In [5]:
from pprint import pprint

pprint(dataset["train"][0])

{'answer': 'Nitrofurantoin',
 'answer_idx': 'D',
 'meta_info': 'step2&3',
 'metamap_phrases': ['23 year old pregnant woman',
                     'weeks presents',
                     'burning',
                     'urination',
                     'states',
                     'started 1 day',
                     'worsening',
                     'drinking',
                     'water',
                     'taking cranberry extract',
                     'feels well',
                     'followed by',
                     'doctor',
                     'pregnancy',
                     'temperature',
                     '97',
                     '36',
                     'blood pressure',
                     'mmHg',
                     'pulse',
                     '80 min',
                     'respirations',
                     'min',
                     'oxygen saturation',
                     '98',
                     'room air',
                     'Physical ex

In [6]:
for i in range(3):
    print("=" * 80)
    pprint(dataset["train"][i])

{'answer': 'Nitrofurantoin',
 'answer_idx': 'D',
 'meta_info': 'step2&3',
 'metamap_phrases': ['23 year old pregnant woman',
                     'weeks presents',
                     'burning',
                     'urination',
                     'states',
                     'started 1 day',
                     'worsening',
                     'drinking',
                     'water',
                     'taking cranberry extract',
                     'feels well',
                     'followed by',
                     'doctor',
                     'pregnancy',
                     'temperature',
                     '97',
                     '36',
                     'blood pressure',
                     'mmHg',
                     'pulse',
                     '80 min',
                     'respirations',
                     'min',
                     'oxygen saturation',
                     '98',
                     'room air',
                     'Physical ex

In [7]:
print(len(dataset["train"]))
# print(len(dataset["validation"]))
print(len(dataset["test"]))

10178
1273


In [8]:
import pandas as pd

df = dataset["train"].to_pandas()

df.head()

,question,answer,options,meta_info,answer_idx,metamap_phrases
0,A 23-year-old pregnant woman at 22 weeks gesta...,Nitrofurantoin,"{'A': 'Ampicillin', 'B': 'Ceftriaxone', 'C': '...",step2&3,D,"[23 year old pregnant woman, weeks presents, b..."
1,A 3-month-old baby died suddenly at night whil...,Placing the infant in a supine position on a f...,{'A': 'Placing the infant in a supine position...,step2&3,A,"[3 month old baby died, night, asleep, mother,..."
2,A mother brings her 3-week-old infant to the p...,Abnormal migration of ventral pancreatic bud,{'A': 'Abnormal migration of ventral pancreati...,step1,A,"[mother, week old infant, pediatrician's offic..."
3,A pulmonary autopsy specimen from a 58-year-ol...,Thromboembolism,"{'A': 'Thromboembolism', 'B': 'Pulmonary ische...",step1,A,"[pulmonary autopsy specimen, 58 year old woman..."
4,A 20-year-old woman presents with menorrhagia ...,Von Willebrand disease,"{'A': 'Hemophilia A', 'B': 'Lupus anticoagulan...",step1,D,"[20 year old woman presents, menorrhagia, past..."


In [9]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10178 entries, 0 to 10177
Data columns (total 6 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   question         10178 non-null  object
 1   answer           10178 non-null  object
 2   options          10178 non-null  object
 3   meta_info        10178 non-null  object
 4   answer_idx       10178 non-null  object
 5   metamap_phrases  10178 non-null  object
dtypes: object(6)
memory usage: 477.2+ KB


In [10]:
df.isnull().sum()

question           0
answer             0
options            0
meta_info          0
answer_idx         0
metamap_phrases    0
dtype: int64

Question

↓

Instruction Builder

↓

Chat Format

↓

SFT

### Data Preparation

In [33]:
from pathlib import Path

# RAW_DIR = Path("../datasets/raw")
PROCESSED_DIR = Path("../datasets/processed")

# RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

CHAT_DIR = Path("../datasets/chat")

CHAT_DIR.mkdir(parents=True, exist_ok=True)

In [12]:
def build_canonical_dataset(split):
    canonical = []

    for idx, sample in enumerate(dataset[split]):

        example = {
            "id": f"{split}_{idx}",

            "task": "clinical_qa",

            "question": sample["question"],

            "options": sample["options"],

            "correct_option": sample["answer_idx"],

            "correct_answer": sample["answer"],

            "meta_info": sample["meta_info"]
        }

        canonical.append(example)

    return canonical

In [28]:
train_canonical = build_canonical_dataset("train")
test_canonical = build_canonical_dataset("test")

In [16]:
import json

def save_json(data, filepath):

    with open(filepath, "w") as f:
        json.dump(data, f, indent=2)

In [ ]:
save_json(
    train_canonical,
    PROCESSED_DIR / "train_canonical.json"
)


save_json(
    test_canonical,
    PROCESSED_DIR / "test_canonical.json"
)

In [35]:
SYSTEM_PROMPT = (
    "You are an expert medical AI assistant. "
    "Carefully analyze the clinical question and select the most appropriate answer."
)

In [36]:
def build_chat_dataset(canonical_dataset):

    chat_dataset = []

    for sample in canonical_dataset:

        options = "\n".join(
            [
                f"{key}. {value}"
                for key, value in sample["options"].items()
            ]
        )

        user_prompt = (
            f"Clinical Question:\n\n"
            f"{sample['question']}\n\n"
            f"Options:\n"
            f"{options}\n\n"
            f"Select the best answer."
        )

        assistant = (
            f"The correct answer is {sample['correct_option']}.\n"
            f"{sample['correct_answer']}"
        )

        chat_dataset.append(

            {
                "id": sample["id"],

                "messages": [

                    {
                        "role": "system",
                        "content": SYSTEM_PROMPT
                    },

                    {
                        "role": "user",
                        "content": user_prompt
                    },

                    {
                        "role": "assistant",
                        "content": assistant
                    }

                ]
            }

        )

    return chat_dataset

In [37]:
train_chat = build_chat_dataset(train_canonical)
test_chat = build_chat_dataset(test_canonical)

In [38]:
from pprint import pprint

pprint(train_chat[0])

{'id': 'train_0',
 'messages': [{'content': 'You are an expert medical AI assistant. Carefully '
                          'analyze the clinical question and select the most '
                          'appropriate answer.',
               'role': 'system'},
              {'content': 'Clinical Question:\n'
                          '\n'
                          'A 23-year-old pregnant woman at 22 weeks gestation '
                          'presents with burning upon urination. She states it '
                          'started 1 day ago and has been worsening despite '
                          'drinking more water and taking cranberry extract. '
                          'She otherwise feels well and is followed by a '
                          'doctor for her pregnancy. Her temperature is 97.7°F '
                          '(36.5°C), blood pressure is 122/77 mmHg, pulse is '
                          '80/min, respirations are 19/min, and oxygen '
                          'saturati

In [39]:
save_json(
    train_chat,
    CHAT_DIR / "train_chat.json"
)

save_json(
    test_chat,
    CHAT_DIR / "test_chat.json"
)

In [41]:
print(f"Train: {len(train_chat)}")
print(f"Test: {len(test_chat)}")

Train: 10178
Test: 1273


In [42]:
from pprint import pprint

pprint(train_chat[0])

{'id': 'train_0',
 'messages': [{'content': 'You are an expert medical AI assistant. Carefully '
                          'analyze the clinical question and select the most '
                          'appropriate answer.',
               'role': 'system'},
              {'content': 'Clinical Question:\n'
                          '\n'
                          'A 23-year-old pregnant woman at 22 weeks gestation '
                          'presents with burning upon urination. She states it '
                          'started 1 day ago and has been worsening despite '
                          'drinking more water and taking cranberry extract. '
                          'She otherwise feels well and is followed by a '
                          'doctor for her pregnancy. Her temperature is 97.7°F '
                          '(36.5°C), blood pressure is 122/77 mmHg, pulse is '
                          '80/min, respirations are 19/min, and oxygen '
                          'saturati

In [44]:
question_lengths = []

for sample in train_canonical:
    question_lengths.append(len(sample["question"]))

import pandas as pd

pd.Series(question_lengths).describe()

count    10178.000000
mean       724.120358
std        275.953423
min         66.000000
25%        527.250000
50%        694.000000
75%        892.000000
max       3577.000000
dtype: float64

In [45]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(
    "Qwen/Qwen2.5-0.5B"
)

token_lengths = []

for sample in train_chat:

    text = tokenizer.apply_chat_template(
        sample["messages"],
        tokenize=False
    )

    tokens = tokenizer(text)["input_ids"]

    token_lengths.append(len(tokens))

In [46]:
pd.Series(token_lengths).describe()

count    10178.000000
mean       266.363922
std         80.273732
min         97.000000
25%        209.000000
50%        254.000000
75%        309.000000
max        996.000000
dtype: float64